# FE Bar — Execution Verification Report

**Generated:** 2026-10-10  
**Purpose:** Proves all code artifacts in this repo actually executed and produced real output.  
**Scope:** Pipelines, App, Dashboard, ML Model

---

## 1. Pipeline Execution Evidence

### Cycle AI Classification Pipeline
- **Pipeline ID:** `bfd04309-db1c-4429-bb3e-3c61fef4ea99`
- **Status:** RUNNING (actively refreshing)
- **20+ updates** executed since creation
- **Last successful full refresh:** Update `2eee0d2a` (COMPLETED)

### Gripper Cycle Count Pipeline
- **Pipeline ID:** `5b456359-3fa5-4ddd-b8ab-66299601c16b`
- **Status:** Deleted (superseded by Cycle AI Classification Pipeline)
- **Tables were dropped** when pipeline was removed

In [1]:
# Verify pipeline output tables exist and contain data
tables = [
    'robot_demo_catalog.robot_demo.silver_cycle_metrics_base',
    'robot_demo_catalog.robot_demo.silver_cycle_metrics',
    'robot_demo_catalog.robot_demo.gold_operational_kpis',
]
for t in tables:
    df = spark.table(t)
    print(f'{t.split(".")[-1]:35s} | rows: {df.count():>6,} | cols: {len(df.columns)}')

silver_cycle_metrics_base            | rows:  1,034 | cols: 12
silver_cycle_metrics                 | rows:  1,034 | cols: 15
gold_operational_kpis                | rows:      7 | cols: 15


In [2]:
# Silver table: AI-classified cycle metrics (sample)
spark.table('robot_demo_catalog.robot_demo.silver_cycle_metrics').select(
    'arm_serial','cycle_index','cycle_time_sec','rotation_type',
    'cycle_quality','shift_name','ai_failure_category','ai_confidence'
).orderBy('cycle_start', ascending=False).show(8, truncate=20)

+-----------+-----------+--------------+-------------+-------------+-----------+-------------------+-------------+
| arm_serial|cycle_index|cycle_time_sec|rotation_type|cycle_quality| shift_name|ai_failure_category|ai_confidence|
+-----------+-----------+--------------+-------------+-------------+-----------+-------------------+-------------+
|5C821097531|          3|          3.62|110 deg extra|          Bad|Night Shift|  calibration_error|         0.78|
|5C821097531|          2|         35.21|110 deg extra|          Bad|Night Shift|  calibration_error|         0.78|
|5C821097531|          1|         32.67|      100 deg|         Good|Night Shift|   normal_operation|         0.78|
|5C821097531|          8|         35.02|      100 deg|         Good|Night Shift|   incomplete_cycle|         0.66|
|5C821097531|          7|         38.23|      100 deg|         Good|Night Shift|   mechanical_drift|         0.78|
|5C821097531|          6|         38.81|      100 deg|         Good|Night Shift|

In [3]:
# Gold table: OEE KPIs per arm per shift
spark.table('robot_demo_catalog.robot_demo.gold_operational_kpis').select(
    'arm_serial','shift_name','total_cycles','good_cycles',
    'defective_cycles','availability_pct','quality_pct','oee_pct'
).show(10, truncate=False)

+-----------+-----------+------------+-----------+----------------+----------------+-----------+-------+
|arm_serial |shift_name |total_cycles|good_cycles|defective_cycles|availability_pct|quality_pct|oee_pct|
+-----------+-----------+------------+-----------+----------------+----------------+-----------+-------+
|5C821097531|Day Shift  |43          |0          |0               |75.05           |0.00       |NULL   |
|5C821097531|Night Shift|158         |36         |122             |4.85            |22.78      |0.46   |
|5C821097531|Swing Shift|516         |191        |256             |52.11           |37.02      |8.05   |
|5C821097531|Swing Shift|71          |42         |26              |2.61            |59.15      |1.46   |
|5C821097531|Night Shift|5           |1          |4               |23.03           |20.00      |3.77   |
|5C821097531|Swing Shift|211         |118        |86              |31.92           |55.92      |16.28  |
|5C821097531|Night Shift|30          |2          |23   

In [4]:
# AI classification distribution across 1,034 cycles
spark.table('robot_demo_catalog.robot_demo.silver_cycle_metrics') \
    .groupBy('ai_failure_category').count() \
    .orderBy('count', ascending=False).show()

+-------------------+-----+
|ai_failure_category|count|
+-------------------+-----+
|   normal_operation|  390|
|   incomplete_cycle|  239|
|   mechanical_drift|  213|
|  calibration_error|  189|
|               NULL|    3|
+-------------------+-----+


## 2. App Deployment Evidence

### gripper-maintenance-alert
- **URL:** https://gripper-maintenance-alert-7474645654545438.aws.databricksapps.com
- **Compute:** ACTIVE (running)
- **Active Deployment:** `01f1c372486d1713bb079b175d374f86`
- **Deploy Status:** SUCCEEDED — "App started successfully"
- **Last Deploy:** 2026-10-08T23:45:03Z
- **Total Deployments:** 20 (all SUCCEEDED)
- **Source Path:** `/Workspace/Users/noel.hoxie@databricks.com/gripper-maintenance-alert`

The Flask app serves 8 API endpoints:
- `/api/metrics` — OEE from gold_operational_kpis
- `/api/telemetry` — Real-time joint position data from so101_snapshots
- `/api/robot_status` — Live robot phase detection
- `/api/hourly_metrics` — Hourly OEE breakdown from silver_cycle_metrics
- `/api/defects` — AI failure category aggregation
- `/api/defect_detail` — Per-defect drill-down with rationales
- `/api/recent_cycles` — Latest cycle events
- `/api/schedules` — CRUD for Lakebase shift_schedule table

## 3. Dashboard Evidence

### Gripper OEE Dashboard (Lakeview)
- **UUID:** `01f1c057fcff1a2bb71a37f98e50e6bf`
- **Datasets:** 2 (gold_gripper_metrics metric view + hourly_oee SQL query)
- **Widgets:** 5 visualizations

| Widget | Type | Dataset |
| --- | --- | --- |
| Avg OEE % | counter | gold_gripper_metrics |
| Average OEE | counter | gold_gripper_metrics |
| Cycle Quality by Arm | bar chart | gold_gripper_metrics |
| OEE Breakdown by Hour | line chart | hourly_oee |
| OEE Metrics Detail | table | gold_gripper_metrics |

Dashboard config includes 11 measures (Total Cycles, Good Cycles, Defective Cycles, Run Time, etc.) and uses `MEASURE()` syntax for aggregations.

## 4. ML Model Evidence

### SO101 Drop Quality ML Model
- **Notebook:** `notebooks/SO101_Drop_Quality_ML_Model.ipynb` (cell outputs embedded)
- **MLflow Experiment:** `/Users/noel.hoxie@databricks.com/SO101_Drop_Quality_ML`
- **Dataset:** 93 cycles × 66 features from `so101_snapshots`

**Execution Results (from notebook cell outputs):**

| Metric | Random Forest | Gradient Boosted Trees |
| --- | --- | --- |
| Accuracy | 96.8% | 96.8% |
| F1 Score | 0.949 | 0.951 |
| AUC-ROC | **0.997** | 0.986 |
| Precision | **1.000** | 0.967 |
| Recall | 0.903 | **0.935** |

The notebook contains:
- Cell 2: Feature engineering output (93 cycles × 66 features, label distribution)
- Cell 3: Model comparison table + MLflow model links
- Cell 4: Feature importance bar chart (embedded PNG) + top-15 features table